In [16]:
import warnings
import os 
import subprocess 

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import scienceplots 
import torch
import torch.distributions as D

# flow_matching
from flow_matching.solver import ODESolver
from matplotlib.gridspec import GridSpec

from model import WrappedModel, vf, prior
from data import generate_quad_gmm

torch.manual_seed(42)
device = torch.device("cpu")

In [ ]:
# 1. Define the folder and filename
folder_path = './checkpoints/' # Change this to your desired folder
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

checkpoint_path = os.path.join(folder_path, 'cond_fm.pth')
ckpt = torch.load(checkpoint_path)
vf.load_state_dict(ckpt["vf_state_dict"])
prior.load_state_dict(ckpt["prior_state_dict"])

In [9]:
wrapped_vf = WrappedModel(vf)
solver = ODESolver(velocity_model=wrapped_vf)  # create an ODESolver class

x_1, y, d, _ = generate_quad_gmm(20000)
# source distribution is an isotropic gaussian
gaussian_log_density = D.Independent(
    D.Normal(torch.zeros(2, device=device), torch.ones(2, device=device)), 1
).log_prob

# compute log likelihood with unbiased hutchinson estimator, average over num_acc

step_size = 0.01
n_steps = 200

# compute with exact divergence
output_cond = solver.compute_likelihood(
    x_1=x_1,
    y=y,
    time_grid=torch.linspace(1, 0, n_steps),
    method="midpoint",
    step_size=None,
    exact_divergence=False,
    log_p0=gaussian_log_density,
    return_intermediates=True,
)
output_uncond = solver.compute_likelihood(
    x_1=x_1,
    y=torch.ones_like(y) * -1,
    time_grid=torch.linspace(1, 0, n_steps),
    method="midpoint",
    step_size=None,
    exact_divergence=False,
    log_p0=gaussian_log_density,
    return_intermediates=True,
)

In [10]:
def plot_flow_results(data, y, d, N=20000, save_path=None):
    fig = plt.figure(figsize=(20, 14))
    gs = GridSpec(3, 3, figure=fig, width_ratios=[1.1, 2.1, 1], height_ratios=[1, 1, 1])
    
    # 1. Main 3D Plot (Center Column)
    ax_3d = fig.add_subplot(gs[:, 1], projection="3d")
    for i in range(0, 200):
        z = np.full(N, (200 - i) / 200)
        pts = data[0][i][:N].detach().numpy()
        ax_3d.scatter(pts[:, 0], pts[:, 1], z, alpha=0.05, s=20, c=y[:N].detach().numpy(), cmap="viridis")

    # Wireframes for 3D
    for t in torch.linspace(0, 1, 3):
        X_p, Y_p = np.meshgrid(np.linspace(-4, 4, 10), np.linspace(-4, 4, 10))
        ax_3d.plot_wireframe(X_p, Y_p, np.full_like(X_p, t), color="gray", alpha=0.5, linewidth=1.0)

    ax_3d.set_xlabel("$x_0$", fontsize=45, labelpad=15)
    ax_3d.set_ylabel("$x_1$", fontsize=45, labelpad=15)
    ax_3d.set_zlabel("t", fontsize=40, labelpad=15)
    ax_3d.tick_params(labelsize=30)
    ax_3d.view_init(elev=20, azim=-65)

    # 2. 2D Snapshots (Left and Right Columns)
    time_indices = [0, 99, 199]
    left_axes = []
    
    for i, t_idx in enumerate(time_indices):
        pts = data[0][t_idx][:N].detach().numpy()
        
        # Right Column (Colored by y)
        ax_r = fig.add_subplot(gs[i, 2])
        ax_r.scatter(pts[:, 0], pts[:, 1], alpha=0.1, s=5, c=y[:N].detach().numpy(), cmap="viridis")
        
        # Left Column (Colored by d)
        ax_l = fig.add_subplot(gs[i, 0])
        sc_l = ax_l.scatter(pts[:, 0], pts[:, 1], s=5, c=d[:N].detach().numpy(), cmap="Greys")
        left_axes.append(ax_l)
        
        for ax in [ax_l, ax_r]:
            ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
            ax.tick_params(axis="both", labelsize=28)

    # Colorbar for the Left Column
    cbar = fig.colorbar(sc_l, ax=left_axes, shrink=0.8, pad=-0.2)
    cbar.set_label("$d$", fontsize=40, loc="top", rotation=0, labelpad=50)
    cbar.ax.tick_params(labelsize=25)

    if save_path: plt.savefig(save_path, dpi=300)
    plt.show()

In [ ]:
plot_flow_results(output_uncond, y, d, N=10000)
# style from paper not included as it requires external latex dependency.

In [ ]:
plot_flow_results(output_cond, y, d, N=10000)
# style from paper not included as it requires external latex dependency.